In [ ]:
import pygmt
import pandas as pd
import matplotlib.pyplot as plt
import verde as vd
import pooch

# ⚠️ Warning!
## This cell will download 40 GB of data

In [ ]:
pooch.retrieve('https://reate.cprm.gov.br/arquivos/index.php/s/z0XoautAuswCSbf/download?path=%2FNAO_SISMICOS%2FMAGNETOMETRIA_GRAVIMETRIA%2F0001_PARANA_28058&files=0001_PARANA_28058_med_proc_01.asc',
               known_hash='md5:4457b4a290aa729e454f1d4221843f1e', fname='0001_PARANA_28058_med_proc_01.asc', progressbar=True)

Given that the original dataset is around 40 Gb, we must apply preprocessing steps such as decimation or chunked loading to handle it efficiently with pandas.

Import the real dataset and decimate it from 100 Hz to 1 Hz.

In [ ]:
# arquivo = '/home/arthur/Documentos/ANP_MAG/0001_PARANA_28058_med_proc_01.asc'
# output_file = "../data/parana-basin-magnetic-processed.csv"
# media_cada = 100
# porcentagem_minima = 0.7
# ncols = 20

# with open(arquivo) as f:
#     with open(output_file, "w") as output:
#         processando = []
#         linha_de_voo_atual = None
#         for i, linha in enumerate(f):
#             # # Tirar o break depois dos testes
#             # if i >= 10000000:
#             #     break
#             if linha.startswith("/"):
#                 output.write(linha.replace("/", "#"))
#                 continue
#             if linha.startswith("FID"):
#                 output.write(linha)
#                 continue
#             partes = linha.strip().split(",")
#             # Tem uma linha com MAGIGRF zuada
#             # try:
#             #     float(partes[17])
#             # except ValueError:
#             #     continue
#             if len(partes) != ncols:
#                 continue
#             linha_de_voo = partes[2]
#             if linha_de_voo_atual is None:
#                 linha_de_voo_atual = linha_de_voo
#             if len(processando) == media_cada or (linha_de_voo != linha_de_voo_atual and len(processando) >= porcentagem_minima * media_cada):
#                 partes_media = []
#                 for j in range(ncols):
#                     if j in {3, 4, 5, 6, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19}:
#                         partes_media.append(f"{np.mean([float(l[j]) for l in processando]):.8f}")
#                     else:
#                         partes_media.append(processando[len(processando) // 2][j])
#                 output.write(",".join(partes_media) + "\n")
#                 processando = []            
#             if linha_de_voo != linha_de_voo_atual:                        
#                 linha_de_voo_atual = linha_de_voo
#                 processando = []
#             processando.append(partes)  

Use pandas to load the real data file, then print and plot it.

In [ ]:
caminho = "../data/parana-basin-magnetic-processed.csv"

df = pd.read_csv(caminho, sep=',', comment="#")
df.columns = ['FID','ESTACAO','LINHA','X','Y','LATITUDE','LONGITUDE','DATA','HORA','GPSALT','BARO','MAGBRU','MAGCOM','MAGBASE','MAGCOR','MAGNIV','MAGMIC','MAGIGRF','IGRF','MDT']

lat = df['LATITUDE']
lon = df['LONGITUDE']
magigrf = df['MAGIGRF']

In [ ]:
df

In [ ]:
fig = pygmt.Figure()

fig.coast(
    region=[-57, -46, -26, -13],
    projection="M20c", 
    frame="afg",
    land="gray"
)

scale = vd.maxabs(df.MAGIGRF)
pygmt.makecpt(cmap="polar+h", series=[-scale, scale])

fig.plot(
    x=df.LONGITUDE,
    y=df.LATITUDE,
    fill=df.MAGIGRF,
    style="c0.05c",
    cmap=True
)

fig.colorbar(position="JBC")
fig.show(width=700)

Identify and remove the tie flight lines from the dataset.

In [ ]:
prefixos = ('19', '29', '39')
df_filtered = df[~df['LINHA'].astype(str).str.startswith(prefixos)].copy()

Using the newly filtered dataset, generate a plot with PyGMT to verify whether the data is now correct.

In [ ]:
fig = pygmt.Figure()

fig.coast(
    region=[-57, -46, -26, -13],
    projection="M20c", 
    frame="afg",
    land="gray"
)

scale = 150
pygmt.makecpt(cmap="polar+h", series=[-scale, scale])

fig.plot(
    x=df_filtered.LONGITUDE,
    y=df_filtered.LATITUDE,
    fill=df_filtered.MAGIGRF,
    style="c0.05c",
    cmap=True
)

fig.colorbar(position="JBC")
fig.show(width=700)